In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [ ]:
train.drop(columns='id', inplace=True)
train.head()

In [ ]:
train.describe

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.replace("http://", "").replace("https://", "")
    text = text.replace("@", "").replace("#", "")
    return text

In [ ]:
train['text'] = train['text'].astype(str).apply(clean_text)
test['text'] = test['text'].astype(str).apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop(columns='target')
y = train['target']
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))
X_tr = vectorizer.fit_transform(X_tr['text'])
X_val = vectorizer.transform(X_val['text'])
X_test = vectorizer.transform(test['text'])

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_tr, y_tr)

In [ ]:
from sklearn.metrics import accuracy_score

val_preds = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_preds))

In [ ]:
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'target': preds})
submission.to_csv('submission.csv', index=False)